### Week 6, Day 2

We're about to create and use our own MCP Server and MCP Client!

It's pretty simple, but it's not super-simple. The excitment around MCP is about how easy it is to share and use other MCP Servers - making our own does involve a bit of work.

Let's review some python code made mostly by a hard-working Engineering Team:

accounts.py

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

True

In [2]:
from accounts import Account

In [3]:
account = Account.get("Jaymineh")
account

Account(name='jaymineh', balance=10000.0, strategy='', holdings={}, transactions=[], portfolio_value_time_series=[])

In [4]:
account.buy_shares("AMZN", 3, "Because this bookstore website looks promising")

'Completed. Latest details:\n{"name": "jaymineh", "balance": 9933.868, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 22.044, "timestamp": "2026-03-30 18:56:30", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-03-30 18:56:30", 10068.868]], "total_portfolio_value": 10068.868, "total_profit_loss": 68.8680000000004}'

In [5]:
account.report()

'{"name": "jaymineh", "balance": 9933.868, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 22.044, "timestamp": "2026-03-30 18:56:30", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-03-30 18:56:30", 10068.868], ["2026-03-30 18:56:32", 10137.868]], "total_portfolio_value": 10137.868, "total_profit_loss": 137.8680000000004}'

In [6]:
account.list_transactions()

[{'symbol': 'AMZN',
  'quantity': 3,
  'price': 22.044,
  'timestamp': '2026-03-30 18:56:30',
  'rationale': 'Because this bookstore website looks promising'}]

### Now we write an MCP server and use it directly!

In [7]:
# Now let's use our accounts server as an MCP server

params = {"command": "uv", "args": ["run", "accounts_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()


In [8]:
mcp_tools

[Tool(name='get_balance', title=None, description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None),
 Tool(name='get_holdings', title=None, description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'integer'}, 'title': 'get_holdingsDictOutput', 'type': 'object'}, icons=None, annotations=None, meta=None),
 Tool(name='buy_shares', 

In [9]:
instructions = "You are able to manage an account for a client, and answer questions about the account."
request = "My name is Jaymineh and my account is under the name Jaymineh. What's my balance and my holdings?"
model = "gpt-5.4-mini"

In [10]:

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("account_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))


Your account balance is **$9,933.87**.

Your holdings are:
- **AMZN:** 3 shares

### Now let's build our own MCP Client

In [11]:
from accounts_client import get_accounts_tools_openai, read_accounts_resource, list_accounts_tools

mcp_tools = await list_accounts_tools()
print(mcp_tools)
openai_tools = await get_accounts_tools_openai()
print(openai_tools)

[Tool(name='get_balance', title=None, description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None), Tool(name='get_holdings', title=None, description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'integer'}, 'title': 'get_holdingsDictOutput', 'type': 'object'}, icons=None, annotations=None, meta=None), Tool(name='buy_shares', ti

In [15]:
request = "My name is Jaymineh and my account is under the name Jaymineh. What's my balance?"

with trace("account_mcp_client"):
    agent = Agent(name="account_manager", instructions=instructions, model=model, tools=openai_tools)
    result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Your cash balance is **$9,933.87**.

In [13]:
context = await read_accounts_resource("ed")
print(context)

{"name": "ed", "balance": 10000.0, "strategy": "", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2026-03-30 19:04:02", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}


In [16]:
from accounts import Account
Account.get("jaymineh").report()

'{"name": "jaymineh", "balance": 9933.868, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 22.044, "timestamp": "2026-03-30 18:56:30", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-03-30 18:56:30", 10068.868], ["2026-03-30 18:56:32", 10137.868], ["2026-03-30 19:07:03", 10080.868]], "total_portfolio_value": 10080.868, "total_profit_loss": 80.8680000000004}'

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Make your own MCP Server! Make a simple function to return the current Date, and expose it as a tool so that an Agent can tell you today's date.<br/>Harder optional exercise: then make an MCP Client, and use a native OpenAI call (without the Agents SDK) to use your tool via your client.
            </span>
        </td>
    </tr>
</table>

### Exercise Solution — Simple Date MCP Server

**Part 1:** Create a `date_server.py` with a `get_current_date` tool and use it with the Agents SDK.  
**Part 2 (Harder):** Create a `date_client.py` and use a native OpenAI call (no Agents SDK) to invoke the tool.

In [17]:
date_params = {"command": "uv", "args": ["run", "date_server.py"]}
async with MCPServerStdio(params=date_params, client_session_timeout_seconds=30) as server:
    date_tools = await server.list_tools()

print("Tools exposed by date_server:")
for t in date_tools:
    print(f"  {t.name}: {t.description.strip()}")

Tools exposed by date_server:
  get_current_date: Return today's date in ISO format (YYYY-MM-DD).

    Use this tool whenever you need to know the current date.


In [18]:
# Part 1: Use date_server.py with the Agents SDK (MCPServerStdio)

date_instructions = "You are a helpful assistant. Use your tools to answer the user's questions accurately."
date_request = "What is today's date?"

async with MCPServerStdio(params=date_params, client_session_timeout_seconds=30) as mcp_server:
    date_agent = Agent(
        name="date_agent",
        instructions=date_instructions,
        model=model,
        mcp_servers=[mcp_server],
    )
    with trace("date_agent"):
        result = await Runner.run(date_agent, date_request)
    display(Markdown(result.final_output))

Today's date is **2026-03-30**.

### Harder Exercise — Native OpenAI Call via MCP Client

Use `date_client.py` to connect to the MCP server and convert its tools into the OpenAI function-calling format.  
Then drive the full tool-use loop yourself with the plain `openai` SDK — **no Agents SDK**.

In [19]:
import json
import openai
from date_client import call_date_tool, get_date_tools_openai_format

# Fetch tools from the MCP server via our own client
openai_date_tools = await get_date_tools_openai_format()
print("Tools in OpenAI format:")
for t in openai_date_tools:
    print(f"  {t['function']['name']}: {t['function']['description'].strip()}")

Tools in OpenAI format:
  get_current_date: Return today's date in ISO format (YYYY-MM-DD).

    Use this tool whenever you need to know the current date.


In [21]:
# Native OpenAI tool-use loop — no Agents SDK involved
client = openai.AsyncOpenAI()

messages = [
    {"role": "system", "content": "You are a helpful assistant with access to tools."},
    {"role": "user", "content": "What is today's date? Please use your tool to find out."},
]

response = await client.chat.completions.create(
    model="gpt-5.4-mini",
    tools=openai_date_tools,
    messages=messages,
)

# Agentic loop: keep going while the model wants to call tools
while response.choices[0].finish_reason == "tool_calls":
    assistant_message = response.choices[0].message
    messages.append(assistant_message)

    tool_results = []
    for tool_call in assistant_message.tool_calls:
        tool_name = tool_call.function.name
        tool_args = json.loads(tool_call.function.arguments)
        print(f"Calling MCP tool '{tool_name}' with args {tool_args}")
        result_text = await call_date_tool(tool_name, tool_args)
        print(f"  → {result_text}")
        tool_results.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result_text,
        })

    messages.extend(tool_results)
    response = await client.chat.completions.create(
        model="gpt-5.4-mini",
        tools=openai_date_tools,
        messages=messages,
    )

final_answer = response.choices[0].message.content
display(Markdown(final_answer))

Calling MCP tool 'get_current_date' with args {}
  → 2026-03-30


Today’s date is **2026-03-30**.